# Experimento: tiempo de carga con 350 filas vs. todas las filas
Archivo: `Julio_BI.csv`

Ejecuta las celdas en orden. Si tu catálogo no se llama `workspace`, cámbialo en todo el notebook.

## Paso 1. Crear la base de datos
Se crea la base de datos y un volumen dentro de ella para guardar el CSV.

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bd_experimento

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bd_experimento.archivos

**Subir el CSV:** menú *Catalog* → `workspace` → `bd_experimento` → `archivos` → botón *Upload to this volume* → elegir `Julio_BI.csv`.

## Paso 2. Listar los archivos con %fs ls
Debe aparecer `Julio_BI.csv`.

In [0]:
%fs ls /Volumes/workspace/bd_experimento/archivos/

[]

## Paso 3. Ver el inicio del archivo con %fs head
Muestra el contenido crudo del CSV (por defecto trae hasta 64 KB, por eso la salida es larga; solo importan las primeras líneas).

In [0]:
%fs head /Volumes/workspace/bd_experimento/archivos/experimentooo/JulioBI.csv

[Truncated to first 65536 bytes]
﻿IDRECETA,IDPACIENTE,IDMEDICO,Suma de CANTIDAD,NOMMEDICAM,Suma de GRAMOSPASTILLA,Suma de DIAPASTILLA,Año,Trimestre,Mes,Día,HORAREGIS,IDCITA,HORAPORTOMAR
R001,P01,M01,20,PARACETAMOL,500,5,2025,Trim. 3,septiembre,15,1899-12-30 09:30:00,C001,08:00
R002,P01,M01,15,IBUPROFENO,400,3,2025,Trim. 3,septiembre,15,1899-12-30 09:30:00,C001,20:00
R003,P02,M02,21,AMOXICILINA,500,7,2025,Trim. 4,octubre,1,1899-12-30 10:30:00,C002,08:00
R004,P03,M03,15,AZITROMICINA,500,5,2025,Trim. 4,noviembre,20,1899-12-30 11:00:00,C003,08:00
R005,P03,M03,10,PARACETAMOL,500,5,2025,Trim. 4,noviembre,20,1899-12-30 11:00:00,C003,14:00
R006,P03,M03,10,OMEPRAZOL,20,10,2025,Trim. 4,noviembre,20,1899-12-30 11:00:00,C003,07:00
R007,P04,M04,10,LORATADINA,10,5,2025,Trim. 4,diciembre,1,1899-12-30 12:00:00,C004,09:00
R008,P04,M04,14,DICLOFENACO,50,7,2025,Trim. 4,diciembre,1,1899-12-30 12:00:00,C004,21:00
R009,P06,M06,1,DEXAMETASONA AMP,8,1,2025,Trim. 4,noviembre,10,1899-12-30 10:00:00,C006,10:00
R

## Paso 4. Crear el DataFrame con spark.read.csv

In [0]:
RUTA = "/Volumes/workspace/bd_experimento/archivos/experimentooo/JulioBI.csv"

df = spark.read.csv(RUTA, header=True, inferSchema=True, encoding="UTF-8")

# El CSV trae un BOM al inicio: se limpia del nombre de la primera columna
df = df.toDF(*[c.replace("\ufeff", "").strip() for c in df.columns])

df.printSchema()
display(df)

root
 |-- IDRECETA: string (nullable = true)
 |-- IDPACIENTE: string (nullable = true)
 |-- IDMEDICO: string (nullable = true)
 |-- Suma de CANTIDAD: integer (nullable = true)
 |-- NOMMEDICAM: string (nullable = true)
 |-- Suma de GRAMOSPASTILLA: integer (nullable = true)
 |-- Suma de DIAPASTILLA: integer (nullable = true)
 |-- Año: integer (nullable = true)
 |-- Trimestre: string (nullable = true)
 |-- Mes: string (nullable = true)
 |-- Día: integer (nullable = true)
 |-- HORAREGIS: timestamp (nullable = true)
 |-- IDCITA: string (nullable = true)
 |-- HORAPORTOMAR: time(6) (nullable = true)



IDRECETA,IDPACIENTE,IDMEDICO,Suma de CANTIDAD,NOMMEDICAM,Suma de GRAMOSPASTILLA,Suma de DIAPASTILLA,Año,Trimestre,Mes,Día,HORAREGIS,IDCITA,HORAPORTOMAR
R001,P01,M01,20,PARACETAMOL,500,5,2025,Trim. 3,septiembre,15,1899-12-30T09:30:00.000Z,C001,08:00:00
R002,P01,M01,15,IBUPROFENO,400,3,2025,Trim. 3,septiembre,15,1899-12-30T09:30:00.000Z,C001,20:00:00
R003,P02,M02,21,AMOXICILINA,500,7,2025,Trim. 4,octubre,1,1899-12-30T10:30:00.000Z,C002,08:00:00
R004,P03,M03,15,AZITROMICINA,500,5,2025,Trim. 4,noviembre,20,1899-12-30T11:00:00.000Z,C003,08:00:00
R005,P03,M03,10,PARACETAMOL,500,5,2025,Trim. 4,noviembre,20,1899-12-30T11:00:00.000Z,C003,14:00:00
R006,P03,M03,10,OMEPRAZOL,20,10,2025,Trim. 4,noviembre,20,1899-12-30T11:00:00.000Z,C003,07:00:00
R007,P04,M04,10,LORATADINA,10,5,2025,Trim. 4,diciembre,1,1899-12-30T12:00:00.000Z,C004,09:00:00
R008,P04,M04,14,DICLOFENACO,50,7,2025,Trim. 4,diciembre,1,1899-12-30T12:00:00.000Z,C004,21:00:00
R009,P06,M06,1,DEXAMETASONA AMP,8,1,2025,Trim. 4,noviembre,10,1899-12-30T10:00:00.000Z,C006,10:00:00
R010,P06,M06,10,PARACETAMOL,500,5,2025,Trim. 4,noviembre,10,1899-12-30T10:00:00.000Z,C006,18:00:00


## Paso 5. Tiempo de carga con 350 filas

In [0]:
import time

t0 = time.perf_counter()
df_350 = spark.read.csv(RUTA, header=True, inferSchema=True, encoding="UTF-8").limit(350)
filas_350 = df_350.count()   # count() obliga a que la carga se ejecute
t_350 = (time.perf_counter() - t0) * 1000

print(f"Filas cargadas: {filas_350} | Tiempo de carga: {t_350:.1f} ms")

Filas cargadas: 350 | Tiempo de carga: 1413.1 ms


## Paso 6. Tiempo de carga con todas las filas

In [0]:
t0 = time.perf_counter()
df_total = spark.read.csv(RUTA, header=True, inferSchema=True, encoding="UTF-8")
filas_total = df_total.count()
t_total = (time.perf_counter() - t0) * 1000

print(f"Filas cargadas: {filas_total} | Tiempo de carga: {t_total:.1f} ms")

Filas cargadas: 999 | Tiempo de carga: 1141.6 ms


## Paso 7. Cuadro de comparación

In [0]:
import pandas as pd

comparacion = pd.DataFrame({
    "Tipo de carga": ["350 filas", "Todas las filas"],
    "Filas": [filas_350, filas_total],
    "Tiempo de carga (ms)": [round(t_350, 1), round(t_total, 1)],
})
comparacion["Diferencia vs 350 filas (ms)"] = (
    comparacion["Tiempo de carga (ms)"] - round(t_350, 1)
).round(1)

display(comparacion)

Tipo de carga,Filas,Tiempo de carga (ms),Diferencia vs 350 filas (ms)
350 filas,350,1413.1,0.0
Todas las filas,999,1141.6,-271.5
